# 01 — Parçalı ileri ve geri yayılım


In [ ]:
import random
import torch
import torch.nn.functional as F

words = open("names.txt", "r").read().splitlines()
chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3

def build_dataset(items):
    X, Y = [], []
    for word in items:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print(Xtr.shape, Xdev.shape, Xte.shape, vocab_size)

In [ ]:
n_embd, n_hidden, batch_size = 10, 64, 32
n = batch_size
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [ ]:
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)
hprebn = embcat @ W1 + b1

bnmeani = hprebn.sum(0, keepdim=True) / n
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = bndiff2.sum(0, keepdim=True) / (n - 1)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
h = torch.tanh(hpreact)
logits = h @ W2 + b2

logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

intermediates = [
    logprobs, probs, counts, counts_sum, counts_sum_inv, norm_logits,
    logit_maxes, logits, h, hpreact, bnraw, bnvar_inv, bnvar,
    bndiff2, bndiff, hprebn, bnmeani, embcat, emb,
]
for tensor in intermediates:
    tensor.retain_grad()
for p in parameters:
    p.grad = None
loss.backward()
print(f"loss: {loss.item():.6f}")

In [ ]:
for name, tensor in zip([
    "logprobs", "probs", "counts", "counts_sum", "counts_sum_inv",
    "norm_logits", "logit_maxes", "logits", "h", "hpreact", "bnraw",
    "bnvar_inv", "bnvar", "bndiff2", "bndiff", "hprebn", "bnmeani",
    "embcat", "emb",
], intermediates):
    print(f"{name:16s} shape={str(tuple(tensor.grad.shape)):16s} |grad|max={tensor.grad.abs().max().item():.6g}")